In [1]:
from experiment_management import (
    create_prompt_group_from_strings,
    init_schema,register_loss_fn,
    ExperimentTemplate,
    persist_experiment_template,
    create_vector,
    persist,
    ExperimentLiveInstance,
    LiveInstance,BatchSteer
)
import json
import torch
import torch.nn.functional as F

init_schema()
#create loss function and register it in application before liveinstances are initialised
def melbo_lesswrong_loss(steered_output, vanilla_output, input_ids, p: int, q: int, target_layer: int):
    steered_hidden = steered_output.hidden_states[target_layer]
    vanilla_hidden = vanilla_output.hidden_states[target_layer]
    delta = steered_hidden - vanilla_hidden
    token_norm = delta.norm(dim=-1).pow(p).sum(dim=1)
    loss = -token_norm.pow(1 / q).sum()
    return loss
register_loss_fn("melbo_lesswrong_loss", melbo_lesswrong_loss)

#helper
def ab_mask_batched(tokens: torch.Tensor, a, b, include_b=True):
    # tokens: [B, L]
    B, L = tokens.shape
    device = tokens.device
    is_a = (tokens == a).to(torch.int32)   # +1 start
    is_b = (tokens == b).to(torch.int32)   # -1 end event
    delta = torch.zeros(B, L + 1, dtype=torch.int32, device=device)
    delta[:, :L] += is_a
    if include_b:
        # subtract at position after b (clamped to sentinel L)
        idx = torch.arange(L, device=device).unsqueeze(0).expand(B, L)
        end_pos = (idx + 1).clamp_max(L)
        delta.scatter_add_(1, end_pos, -is_b)
    else:
        delta[:, :L] -= is_b
    return delta[:, :-1].cumsum(dim=1) > 0   # [B, L] bool


prompts = ['Write a reciepe for onion soup',
]
prompt_chat_format = [[
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
        ],
    },
] for prompt in prompts]


In [2]:

group = create_prompt_group_from_strings(json.dumps(prompt_chat) for prompt_chat in prompt_chat_format)
group_id = group.group_id

loss_kwargs = {"p": 2, "q": 2, "target_layer": -1}
loss_kwargs_json = json.dumps(loss_kwargs)
optimizer_kwargs = {"lr": 1e-3}
optimizer_kwargs_json = json.dumps(optimizer_kwargs)
et = persist_experiment_template(ExperimentTemplate(
    prompt_group=group_id,
    loss_name="melbo_lesswrong_loss",
    loss_additional_parameters=loss_kwargs_json,
    optimizer_name="AdamW",
    optimizer_additional_parameters=optimizer_kwargs_json,
    module_name="model.language_model.layers.6",
    batch_size=1,
    normalization=1.0,
))

<class 'experiment_management.orm.Prompt'> Exists, returning that
Exact same group exists, returning that
Experiment Exists, returning that


In [3]:
# #model loading
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend
model_id = "DominicJW/Ministral-3-3B-Instruct-2512-BF16-bnb-4bit"
model_id = "/home/u/.cache/huggingface/hub/models--DominicJW--Ministral-3-3B-Instruct-2512-BF16-bnb-4bit/snapshots/43aec86bebdbb2a86e6c5878123a0fdf29e48612"
tokenizer = MistralCommonBackend.from_pretrained(model_id, local_files_only=True)
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id, device_map="cuda", local_files_only=True
)
for param in model.parameters():
    param.requires_grad = False

vector_width = model.config.text_config.hidden_size
vector_shape = (vector_width,)

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

In [6]:
et_1 = et
seeds = [58203,14324,95232,85312,45925] #starting with the same vectors so we can see how normalization changes it
vecs = [create_vector(seed=seed,shape=vector_shape) for seed in seeds]



prompts = [link.prompt for link in group.group_links]
batch_prompts = [json.loads(prompt.text) for prompt in prompts]

vanilla_inputs = tokenizer.apply_chat_template(batch_prompts, return_tensors="pt", padding=True).to(model.device)
vanilla_input_ids = vanilla_inputs.input_ids
vanilla_attention_mask = vanilla_inputs.attention_mask

steered_input_ids = vanilla_input_ids.repeat(len(vecs), 1)
steered_attention_mask = vanilla_attention_mask.repeat(len(vecs), 1)

for normalization_value in range(1,15):
    et_1.normalization = float(normalization_value)
    et_1 = persist(et_1)

    lives = [persist(ExperimentLiveInstance(experiment_template=et_1,initial_vector=vec,vector_data=vec.vector_data),check_dupe = "warn") for vec in vecs]

    live_objs = [LiveInstance(live) for live in lives]
    batch_steer = BatchSteer(live_objs, model)

    with torch.no_grad(), batch_steer.steer():
        generated_ids = model.generate(
            input_ids=steered_input_ids,
            attention_mask=steered_attention_mask,
            max_new_tokens=30,#enough context to judge sanity of response
        )[:,vanilla_input_ids.shape[1]:]

        eos = (generated_ids == 2).any().item()
        generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    #proccessing generated text
    for i, text in enumerate(generated_text):
        prompt_id = prompts[i % len(prompts)].prompt_id
        live_index = i // len(prompts)
        live_obj = live_objs[live_index]


        text_as_assistant_response = [
        {"role": "assistant","content": [{"type": "text", "text": text},],},]


        #printing responses out
        screen_width = 200
        if (i % len(prompts)) == 0:
            title = f"Initial Vector: {live_obj.live.initial_vector_id} Normalization: {live_obj.live.experiment_template.normalization}" 
            space = f" "
            dash = f"-"
            print(f"#{dash.center(screen_width,dash)}#")
            print(f"#{space.center(screen_width)}#")
            print(f"#{title.center(screen_width)}#")
            print(f"#{space.center(screen_width)}#")
            print(f"#{dash.center(screen_width,dash)}#")
        else:
            print(f"#{dash.center(screen_width,dash)}#")
        title = f"Prompt: {prompt_id}"
        print(f"#{title.center(screen_width)}#")
        print(f"#{dash.center(screen_width,dash)}#")
        print()
        print(text)

vector of matching seed and shape exists, returning that vector
vector of matching seed and shape exists, returning that vector
vector of matching seed and shape exists, returning that vector
vector of matching seed and shape exists, returning that vector
vector of matching seed and shape exists, returning that vector
<class 'experiment_management.orm.ExperimentTemplate'> Exists, returning that
#--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#                                                                                                                                                                                                        #
#                                                                                  Initial Vector: 1 Normalization: 1.0                                                                             

In [ ]:
#See TomatoSoupAnalysis for details
#From the output I will briefly say that as the vectors increase in magnitude, the outputs become more incoherent.



In [8]:
generated_ids = model.generate(
    input_ids=vanilla_input_ids,
    attention_mask=vanilla_attention_mask,
    max_new_tokens=30,
)[:,vanilla_input_ids.shape[1]:]
eos = (generated_ids == 2).any().item()
generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

In [9]:
print("Vanilla generated text:")
for i, text in enumerate(generated_text):
    text_as_assistant_response = [{"role": "assistant","content": [{"type": "text", "text": text},],},]
    prompt_id = prompts[i % len(prompts)].prompt_id
    print(text)

Vanilla generated text:
Here’s a classic **Onion Soup** recipe that’s hearty, flavorful, and perfect for a cozy meal. This version includes


In [ ]:
#input the prompt, but not include it in the calculation (as models are not trained on next word prediction of the prompt)
user_prompt = json.loads(prompt_dtos[0].text)
assistant_response = json.loads(vanilla_out.text)
chat_input = user_prompt + assistant_response
inputs = tokenizer.apply_chat_template(chat_input, return_tensors="pt", padding=True,continue_final_message=True).to(model.device)

with torch.no_grad():
    logits = model(**inputs).logits

logits = logits[:,:-1]
targets = inputs["input_ids"][:,1:]

log_probs = F.log_softmax(logits, dim=-1)          # (B, L, V)

# gather log-prob of the correct token for each position
token_log_probs = log_probs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)  # (B, L)

#create mask
mask = ~ab_mask_batched(targets,a=3,b=4)
if True or vanilla_out.dto:
    mask[:,-1] = False

token_log_probs = token_log_probs * mask
# sum negative log-likelihood and normalize by number of tokens
nll = -token_log_probs.sum()
ntokens = mask.sum().clamp_min(1).float()
avg_nll = nll / ntokens
perplexity = torch.exp(avg_nll)

In [ ]:
#Get some preliminary mesurements
#perplexity on self generated
#vanilla on vanilla text
#steered on steered text (same steered)


#cross entropy (or similar (ideally a commutative metric))
#vanilla logits x steered logits on vanilla text
#vanilla logits x steered logits on steered text

In [ ]:
#training loop
vanilla_output = model(**vanilla_inputs, output_hidden_states=True)
for step in range(1, 101):
    with batch_steer.steer():
        steered_output = model(
            input_ids=steered_input_ids,
            attention_mask=steered_attention_mask,
            output_hidden_states=True,
        )
    loss = batch_steer.calc_loss(steered_output, vanilla_output, vanilla_inputs.input_ids)
    loss.backward()
    if step % 5 == 0:
        save_vectors_flag = step % 10 == 0
        save_generated_text_flag = save_vectors_flag #can change
        _ = [live_obj.create_snapshot(save_vector=save_vectors_flag) for live_obj in live_objs]
        for live_obj in live_objs:
            if live_obj.last_training_loss is not None:
                metric = metric_repo.create_from_snapshot(
                    live_obj.last_snapshot,
                    value=float(live_obj.last_training_loss.detach().item()),
                    description="Training Loss",
                )
        if save_vectors_flag:
            with batch_steer.steer():                
                generated_ids = model.generate(
                    input_ids=steered_input_ids,
                    attention_mask=steered_attention_mask,
                    max_new_tokens=40,
                )[:,vanilla_input_ids.shape[1]:]
                generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)
                for i, text in enumerate(generated_text):
                    live_idx = i // len(prompt_dtos)
                    steered_output = output_repo.create_from_snapshot(
                        live_objs[live_idx].last_snapshot,
                        text,
                        prompt_id=prompt_dtos[i % len(prompt_dtos)].prompt_id,
                        vanilla=False,
                    )
    for live_obj in live_objs:
        live_obj.step_optimizer()
        live_obj.iteration_count += 1
        live_obj.update_repo()